## Preparing data

In [ ]:
from preprocessing_Data import Data
import json

In [ ]:
pdf_file = "36-2024-qh15.pdf" 
pdf_file2 = "36-2024-qh15_tiep.pdf"
dator = Data(pdf_path=pdf_file)
dator2 = Data(pdf_path=pdf_file2)
try:
    data1 = dator.preprocess_legal_pdf()
    data2 = dator2.preprocess_legal_pdf()
    data = data1 + data2
    with open("data_structured_granular.json", "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=4)
        
    print(f"Thành công! Đã tách được {len(data)} mẩu dữ liệu nhỏ (Khoản).")
except Exception as e:
    print(f"Lỗi: {e}")
print(data)

In [ ]:
from langchain_core.documents import Document
langchain_docs = []

for item in data:
    doc = Document(
        page_content=item["content"], 
        metadata=item["metadata"]      
    )
    langchain_docs.append(doc)

print(f"Đã chuyển đổi {len(langchain_docs)} tài liệu sang định dạng LangChain.")


In [ ]:
print(langchain_docs[1])

## Khởi tạo database


In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("GROQ")
if api_key:
    print("Đã tìm thấy Grok Api Key!")

db_name = "vector_db"

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma


embeddings = HuggingFaceEmbeddings(model_name="bkai-foundation-models/vietnamese-bi-encoder")


if os.path.exists(db_name) and os.listdir(db_name):
    print("--- Đang kết nối với Vector Database có sẵn... ---")
    vector_db = Chroma(
        persist_directory=db_name, 
        embedding_function=embeddings
    )
else:
    print("--- Không tìm thấy database. Đang nạp dữ liệu lần đầu... ---")
    vector_db = Chroma.from_documents(
        documents=langchain_docs,
        embedding=embeddings,
        persist_directory=db_name
    )
    print("--- Đã tạo và lưu xong dữ liệu vào ChromaDB! ---")

In [ ]:
print(vector_db._collection.count())

In [ ]:
query = "Cho tôi biết nồng độ cồn bao nhiêu thì bị phạt?"


docs = vector_db.similarity_search(query, k=3)

for i, doc in enumerate(docs):
    print(f"\n--- Kết quả {i+1} ---")
    print(f"Nội dung: {doc.page_content}")
    print(f"Nguồn: {doc.metadata.get('article', 'N/A')}")

In [ ]:
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever 
from langchain_classic.chains import create_retrieval_chain 
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableBranch, RunnableLambda
from langchain_groq import ChatGroq

bm25_retriever = BM25Retriever.from_documents(langchain_docs)
bm25_retriever.k = 5


vector_retriever = vector_db.as_retriever(search_kwargs={"k": 5})

hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever],
    weights=[0.3, 0.7] 
)

llm = ChatGroq(
    model="llama-3.3-70b-versatile", 
    groq_api_key=api_key,
    temperature=0.3
)

system_prompt = (
    "Bạn là một chuyên gia tư vấn luật pháp chuyên nghiệp, am hiểu sâu sắc về "
    "Luật Trật tự, an toàn giao thông đường bộ 2024. Nhiệm vụ của bạn là giải đáp "
    "thắc mắc của người dân dựa trên các đoạn trích luật được cung cấp dưới đây:"
    "\n\n"
    "--- DỮ LIỆU LUẬT CĂN CỨ ---\n"
    "{context}\n"
    "---------------------------\n\n"
    "HƯỚNG DẪN TRẢ LỜI:\n"
    "1. TƯ DUY LOGIC: Hãy hiểu rằng các cụm từ đời thường như 'uống rượu', 'uống bia', "
    "'nhậu', 'say xỉn' đều liên quan trực tiếp đến quy định về 'nồng độ cồn'. Nếu luật cấm "
    "điều khiển xe khi có nồng độ cồn, hãy khẳng định việc uống rượu bia rồi lái xe là vi phạm.\n\n"
    "2. CẤU TRÚC PHẢN HỒI:\n"
    "   - Câu đầu tiên: Trả lời trực tiếp vào vấn đề (Có bị phạt không? Có vi phạm không?).\n"
    "   - Phần giải thích: Trích dẫn rõ tên Điều, Khoản và nội dung cốt lõi của quy định đó.\n"
    "   - Phần kết luận: Tóm tắt ngắn gọn lời khuyên hoặc mức phạt (nếu có thông tin).\n\n"
    "3. NGUYÊN TẮC CHÍNH XÁC: Chỉ trả lời dựa trên dữ liệu được cung cấp. "
    "Nếu dữ liệu không đề cập đến một hành vi cụ thể, hãy nói: 'Dựa trên tài liệu hiện có, "
    "chưa có quy định cụ thể về vấn đề này'. Tuyệt đối không tự bịa số Điều hoặc nội dung luật.\n\n"
    "Hãy trả lời bằng tông giọng chuyên nghiệp, điềm đạm và dễ hiểu."
)
prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(hybrid_retriever, question_answer_chain)


system_prompt_advice = (
    "Bạn là trợ lý giao thông tâm lý. Khi người dùng xin lời khuyên hành vi hoặc xử lý tình huống: "
    "Hãy tư vấn các giải pháp an toàn, thực tế (như bắt taxi, gọi người thân, chờ tỉnh táo). "
    "Dùng ngôn ngữ đời thường, nhẹ nhàng. Không cần trích dẫn điều luật khô khan."
)



prompt_advice = ChatPromptTemplate.from_messages([
    ("system", system_prompt_advice),
    ("human", "{input}"),
])

advice_chain = prompt_advice | llm | StrOutputParser()



classifier_prompt = ChatPromptTemplate.from_template(
    "Phân loại câu hỏi sau vào 'LAW' (hỏi số liệu, mức phạt, điều luật) "
    "hoặc 'ADVICE' (hỏi lời khuyên, cách xử lý, tán gẫu).\n"
    "Chỉ trả ra duy nhất từ 'LAW' hoặc 'ADVICE'.\n\n"
    "Câu hỏi: {input}"
)
classifier_chain = classifier_prompt | llm | StrOutputParser()



def wrap_rag(input_data):
    res = rag_chain.invoke({"input": input_data["input"]})
    return res["answer"]



router_branch = RunnableBranch(
    (lambda x: "LAW" in x["topic"].upper(), RunnableLambda(wrap_rag)),
    advice_chain
)



full_chain = (
    {"topic": classifier_chain, "input": lambda x: x["input"]}
    | router_branch
)




query = "Hello"
response = full_chain.invoke({"input": query})

print(f"Câu hỏi: {query}")
print("-" * 30)
print(f"Trả lời: {response}")

In [ ]:
query = "Mình lỡ uống vài ly rồi, giờ có nên lái xe về không?"
response = full_chain.invoke({"input": query})

print(f"Câu hỏi: {query}")
print("-" * 30)
print(f"Trả lời: {response}")

In [ ]:
import gradio as gr



def Chat(query, history):
    response = rag_chain.invoke({"input": query})
    return response["answer"]


demo = gr.ChatInterface(
    Chat
)

demo.launch(share="true")